# Transaction Fraud Transformer Lab

This notebook trains a Transformer-first fraud detector on IEEE-CIS and compares it with practical baselines.

## Experiment Flow

1. Confirm Kaggle input and GPU
2. Build one shared IEEE-CIS sample
3. Train LightGBM as a benchmark
4. Train a T4-safe FT-Transformer
5. Save Transformer artifacts
6. Optionally run pretrained tabular foundation-model benchmarks

The Transformer is the main project model. LightGBM, TabPFN, TabICL, and TabFM are comparison points.


## 1. Setup

Run this section first. It keeps paths, seeds, and artifact locations explicit.


In [ ]:
# === Core imports ===
import gc
import json
import os
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import average_precision_score, f1_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

ARTIFACT_DIR = Path("/kaggle/working/artifacts")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)


In [ ]:
# === Discover Kaggle dataset path ===
CANDIDATE_DATA_DIRS = [
    "/kaggle/input/competitions/ieee-fraud-detection",
    "/kaggle/input/ieee-fraud-detection",
]

DATA_DIR = None
for candidate in CANDIDATE_DATA_DIRS:
    if os.path.exists(os.path.join(candidate, "train_transaction.csv")):
        DATA_DIR = candidate
        break

if DATA_DIR is None:
    print("Available /kaggle/input paths:")
    for root, dirs, files in os.walk("/kaggle/input"):
        print(root)
    raise FileNotFoundError("Attach the IEEE-CIS Fraud Detection input, then rerun this cell.")

print("Using DATA_DIR:", DATA_DIR)
print(os.listdir(DATA_DIR))


In [ ]:
# === GPU and memory check ===
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Torch:", torch.__version__)
print("Device:", DEVICE)

if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    print("Allocated GB:", round(torch.cuda.memory_allocated() / 1e9, 2))
    print("Reserved GB:", round(torch.cuda.memory_reserved() / 1e9, 2))
else:
    print("Turn on Kaggle GPU T4 before training the Transformer.")


## 2. Shared IEEE-CIS Sample

This shared sample keeps every benchmark comparable. The default settings are intentionally conservative after the T4 CUDA out-of-memory error.


In [ ]:
# === T4-safe experiment controls ===
SAMPLE_ROWS = 50_000
BATCH_SIZE = 64
VALIDATION_SIZE = 0.20

TRANSFORMER_EPOCHS = 5
D_MODEL = 64
N_HEAD = 4
N_LAYERS = 2
DROPOUT = 0.10
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 1e-4
EARLY_STOPPING_PATIENCE = 2


In [ ]:
# === Load and join IEEE-CIS ===
train_tx = pd.read_csv(f"{DATA_DIR}/train_transaction.csv")
train_id = pd.read_csv(f"{DATA_DIR}/train_identity.csv")
raw_df = train_tx.merge(train_id, on="TransactionID", how="left")

if SAMPLE_ROWS and SAMPLE_ROWS < len(raw_df):
    df = raw_df.sample(n=SAMPLE_ROWS, random_state=SEED).copy()
else:
    df = raw_df.copy()

print("Rows:", len(df))
print("Columns:", df.shape[1])
print("Fraud rate:", round(float(df["isFraud"].mean()), 4))


In [ ]:
# === Split labels and feature types ===
y = df["isFraud"].astype("float32").values
X = df.drop(columns=["isFraud", "TransactionID"])

cat_cols = X.select_dtypes(include=["object", "string"]).columns.tolist()
num_cols = [col for col in X.columns if col not in cat_cols]

print("Numeric features:", len(num_cols))
print("Categorical features:", len(cat_cols))
print("Total features:", X.shape[1])


## 3. LightGBM Benchmark

LightGBM is the classic tabular benchmark. It tells us what score the Transformer is chasing, but it is not replacing the Transformer in this project.


In [ ]:
# === Train LightGBM baseline ===
from lightgbm import LGBMClassifier

X_lgbm = X.copy()
for col in cat_cols:
    X_lgbm[col] = X_lgbm[col].astype("category")

X_train_lgbm, X_val_lgbm, y_train_lgbm, y_val_lgbm = train_test_split(
    X_lgbm,
    y,
    test_size=VALIDATION_SIZE,
    random_state=SEED,
    stratify=y,
)

lgbm_model = LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=64,
    subsample=0.8,
    colsample_bytree=0.8,
    class_weight="balanced",
    random_state=SEED,
    n_jobs=-1,
)

lgbm_model.fit(X_train_lgbm, y_train_lgbm, categorical_feature=cat_cols)
lgbm_pred = lgbm_model.predict_proba(X_val_lgbm)[:, 1]

lgbm_metrics = {
    "model": "lightgbm_baseline",
    "rows": int(len(df)),
    "features": int(X.shape[1]),
    "roc_auc": float(roc_auc_score(y_val_lgbm, lgbm_pred)),
    "auprc": float(average_precision_score(y_val_lgbm, lgbm_pred)),
}

lgbm_metrics


In [ ]:
# === Save LightGBM benchmark ===
import joblib

joblib.dump(lgbm_model, ARTIFACT_DIR / "lightgbm_baseline.pkl")
with open(ARTIFACT_DIR / "lightgbm_metrics.json", "w") as f:
    json.dump(lgbm_metrics, f, indent=2)

print(lgbm_metrics)


## 4. FT-Transformer Main Model

The Transformer turns each transaction field into a token. Attention then learns interactions between fields, such as amount, device, product code, and timing.


In [ ]:
# === Clean, encode, and scale for Transformer ===
# Split raw rows first; validation never fits an encoder, median, or scaler.
train_idx, val_idx = train_test_split(
    np.arange(len(X)), test_size=VALIDATION_SIZE, random_state=SEED, stratify=y,
)
X_train_raw, X_val_raw = X.iloc[train_idx].copy(), X.iloc[val_idx].copy()
y_train, y_val = y[train_idx], y[val_idx]
numeric_medians = X_train_raw[num_cols].replace([np.inf, -np.inf], np.nan).median().fillna(0)

def clean_numeric(frame):
    return frame[num_cols].replace([np.inf, -np.inf], np.nan).fillna(numeric_medians)

encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
Xc_train = encoder.fit_transform(X_train_raw[cat_cols].fillna("__missing__").astype(str)).astype("int64") + 1
Xc_val = encoder.transform(X_val_raw[cat_cols].fillna("__missing__").astype(str)).astype("int64") + 1
scaler = StandardScaler()
Xn_train = scaler.fit_transform(clean_numeric(X_train_raw)).astype("float32")
Xn_val = scaler.transform(clean_numeric(X_val_raw)).astype("float32")
assert np.isfinite(Xn_train).all() and np.isfinite(Xn_val).all()
assert set(train_idx).isdisjoint(val_idx)
category_cardinalities = [len(values) + 1 for values in encoder.categories_]
# Keep tensors on CPU; move only one batch at a time to the GPU.
train_ds = TensorDataset(torch.from_numpy(Xn_train), torch.from_numpy(Xc_train), torch.from_numpy(y_train))
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_num, val_cat = torch.from_numpy(Xn_val), torch.from_numpy(Xc_val)
print("Train / validation:", len(y_train), len(y_val), "Batch:", BATCH_SIZE)


In [ ]:
# === Define FT-Transformer ===
class FTTransformer(nn.Module):
    def __init__(self, n_num, cat_cards, d_model=64, nhead=4, layers=2, dropout=0.1):
        super().__init__()
        self.num_weight = nn.Parameter(torch.randn(n_num, d_model) * 0.02)
        self.num_bias = nn.Parameter(torch.zeros(n_num, d_model))
        self.cat_embs = nn.ModuleList([nn.Embedding(card, d_model) for card in cat_cards])
        self.cls = nn.Parameter(torch.zeros(1, 1, d_model))

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=4 * d_model,
            dropout=dropout,
            batch_first=True,
            activation="gelu",
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=layers)
        self.head = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, 1),
        )

    def forward(self, x_num, x_cat):
        # Numeric columns become tokens through learned scale and bias parameters.
        num_tokens = x_num.unsqueeze(-1) * self.num_weight.unsqueeze(0) + self.num_bias.unsqueeze(0)
        cat_tokens = [emb(x_cat[:, i]).unsqueeze(1) for i, emb in enumerate(self.cat_embs)]
        tokens = torch.cat([num_tokens] + cat_tokens, dim=1)
        cls = self.cls.expand(tokens.size(0), -1, -1)
        encoded = self.encoder(torch.cat([cls, tokens], dim=1))
        return self.head(encoded[:, 0]).squeeze(-1)


In [ ]:
# === Initialize Transformer ===
gc.collect()
if DEVICE == "cuda":
    torch.cuda.empty_cache()

model_t = FTTransformer(
    n_num=len(num_cols),
    cat_cards=category_cardinalities,
    d_model=D_MODEL,
    nhead=N_HEAD,
    layers=N_LAYERS,
    dropout=DROPOUT,
).to(DEVICE)

optimizer = torch.optim.AdamW(model_t.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=1)

pos_weight = torch.tensor([(len(y_train) - y_train.sum()) / max(y_train.sum(), 1)], device=DEVICE)
loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

print("Model parameters:", sum(p.numel() for p in model_t.parameters()))


In [ ]:
# === Train Transformer with early stopping ===
best_ap = -1.0
best_auc = -1.0
best_epoch = 0
best_state = None
best_pred = None
bad_epochs = 0
history = []

for epoch in range(TRANSFORMER_EPOCHS):
    model_t.train()
    losses = []

    for xb_num, xb_cat, yb in train_loader:
        xb_num = xb_num.to(DEVICE)
        xb_cat = xb_cat.to(DEVICE)
        yb = yb.to(DEVICE)

        optimizer.zero_grad(set_to_none=True)
        logits = model_t(xb_num, xb_cat)
        loss = loss_fn(logits, yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_t.parameters(), 1.0)
        optimizer.step()
        losses.append(float(loss.item()))

    model_t.eval()
    with torch.no_grad():
        # Bound attention memory during validation as well as training.
        parts = []
        for start in range(0, len(val_num), BATCH_SIZE):
            logits = model_t(val_num[start:start+BATCH_SIZE].to(DEVICE),
                             val_cat[start:start+BATCH_SIZE].to(DEVICE))
            parts.append(torch.sigmoid(logits).cpu().numpy())
        pred_t = np.concatenate(parts)

    ap = float(average_precision_score(y_val, pred_t))
    auc = float(roc_auc_score(y_val, pred_t))
    epoch_metrics = {
        "epoch": epoch + 1,
        "loss": float(np.mean(losses)),
        "roc_auc": auc,
        "auprc": ap,
    }
    history.append(epoch_metrics)
    scheduler.step(ap)

    print(
        "epoch", epoch + 1,
        "loss", round(epoch_metrics["loss"], 4),
        "ROC-AUC", round(auc, 4),
        "AUPRC", round(ap, 4),
    )

    if ap > best_ap:
        best_ap = ap
        best_auc = auc
        best_epoch = epoch + 1
        best_state = {k: v.detach().cpu().clone() for k, v in model_t.state_dict().items()}
        best_pred = pred_t
        bad_epochs = 0
    else:
        bad_epochs += 1
        if bad_epochs >= EARLY_STOPPING_PATIENCE:
            print("Early stopping after", epoch + 1, "epochs")
            break

if best_state is not None:
    model_t.load_state_dict(best_state)

print("Best epoch:", best_epoch)
print("Best AUPRC:", round(best_ap, 4))


In [ ]:
# === Save Transformer artifacts ===
torch.save(
    {
        "model_state": model_t.state_dict(),
        "num_numeric_features": len(num_cols),
        "category_cardinalities": category_cardinalities,
        "d_model": D_MODEL,
        "nhead": N_HEAD,
        "num_layers": N_LAYERS,
        "dropout": DROPOUT,
        "best_epoch": best_epoch,
    },
    ARTIFACT_DIR / "model.pt",
)
joblib.dump(encoder, ARTIFACT_DIR / "ft_transformer_encoder.pkl")
joblib.dump(scaler, ARTIFACT_DIR / "ft_transformer_scaler.pkl")

transformer_metrics = {
    "model": "ft_transformer_t4_safe",
    "rows": int(len(df)),
    "best_epoch": int(best_epoch),
    "roc_auc": float(best_auc),
    "auprc": float(best_ap),
    "numeric_features": len(num_cols),
    "categorical_features": len(cat_cols),
    "history": history,
}

with open(ARTIFACT_DIR / "metrics.json", "w") as f:
    json.dump(transformer_metrics, f, indent=2)

print(transformer_metrics)

# Preserve feature order and imputation values for inference.
joblib.dump({"numeric_features": num_cols, "categorical_features": cat_cols,
             "numeric_medians": numeric_medians, "encoder": encoder, "scaler": scaler},
            ARTIFACT_DIR / "preprocessing.pkl")
np.savez(ARTIFACT_DIR / "validation_predictions.npz",
         transaction_ids=df.iloc[val_idx]["TransactionID"].to_numpy(),
         y_true=y_val, probabilities=best_pred)


In [ ]:
# === Verify saved artifacts and inference round trip ===
expected_files = ["model.pt", "metrics.json", "ft_transformer_encoder.pkl",
                  "ft_transformer_scaler.pkl", "preprocessing.pkl",
                  "validation_predictions.npz", "lightgbm_baseline.pkl", "lightgbm_metrics.json"]
for name in expected_files:
    assert (ARTIFACT_DIR / name).is_file(), name
saved_pre = joblib.load(ARTIFACT_DIR / "preprocessing.pkl")
checkpoint = torch.load(ARTIFACT_DIR / "model.pt", map_location="cpu", weights_only=True)
restored = FTTransformer(checkpoint["num_numeric_features"], checkpoint["category_cardinalities"],
                         checkpoint["d_model"], checkpoint["nhead"], checkpoint["num_layers"], checkpoint["dropout"])
restored.load_state_dict(checkpoint["model_state"])
restored.eval()
probe = X.iloc[val_idx[:16]]
pn = saved_pre["scaler"].transform(probe[num_cols].replace([np.inf, -np.inf], np.nan)
                                  .fillna(saved_pre["numeric_medians"])).astype("float32")
pc = saved_pre["encoder"].transform(probe[cat_cols].fillna("__missing__").astype(str)).astype("int64") + 1
with torch.no_grad():
    restored_pred = torch.sigmoid(restored(torch.from_numpy(pn), torch.from_numpy(pc))).numpy()
np.testing.assert_allclose(restored_pred, best_pred[:16], rtol=1e-4, atol=1e-5)
assert set(train_idx).isdisjoint(val_idx)
assert np.isfinite(Xn_train).all() and np.isfinite(Xn_val).all()
del restored
print("PASS: artifacts exist, preprocessing is finite, splits are disjoint, and saved-model predictions match.")


## 5. Foundation Model Benchmarks

These are local pretrained classifiers; transaction rows are not sent to a hosted inference API.
Internet is needed for packages and model weights. Each model is released before the next runs.

The resource-limited comparison uses 2,000 training rows, 1,000 held-out rows, and up to
100 features selected using training data only. LightGBM is refitted on exactly the same
matrix for a matched baseline. Do not compare this score directly with the 50,000-row run.

Failures are recorded with their reason. TabFM remains unimplemented and is not a benchmark result.


In [ ]:
# === Shared foundation-model benchmark slice ===
import importlib.metadata
import subprocess
import sys
import time
from sklearn.feature_selection import SelectKBest, f_classif

# Install explicitly instead of silently skipping missing packages.
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "tabicl==2.2.0", "tabpfn==8.5.0"])
print({name: importlib.metadata.version(name) for name in ("tabicl", "tabpfn")})

# Reconstruct the SAME outer split even when running only the benchmark section.
train_idx, val_idx = train_test_split(np.arange(len(X)), test_size=VALIDATION_SIZE,
                                    random_state=SEED, stratify=y)
def small_subset(indices, limit):
    if len(indices) <= limit:
        return indices
    selected, _ = train_test_split(indices, train_size=limit, random_state=SEED, stratify=y[indices])
    return selected

f_train_idx = small_subset(train_idx, 2000)
f_val_idx = small_subset(val_idx, 1000)
assert set(f_train_idx).isdisjoint(f_val_idx)
ft, fv = X.iloc[f_train_idx].copy(), X.iloc[f_val_idx].copy()
y_train_f, y_val_f = y[f_train_idx].astype(int), y[f_val_idx].astype(int)
f_medians = ft[num_cols].replace([np.inf, -np.inf], np.nan).median().fillna(0)
foundation_encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
tc = foundation_encoder.fit_transform(ft[cat_cols].fillna("__missing__").astype(str))
vc = foundation_encoder.transform(fv[cat_cols].fillna("__missing__").astype(str))
def numeric_f(frame):
    return frame[num_cols].replace([np.inf, -np.inf], np.nan).fillna(f_medians).to_numpy()
train_matrix = np.concatenate([numeric_f(ft), tc], axis=1).astype("float32")
val_matrix = np.concatenate([numeric_f(fv), vc], axis=1).astype("float32")
nonconstant = np.ptp(train_matrix, axis=0) > 0
selector = SelectKBest(f_classif, k=min(100, int(nonconstant.sum())))
X_train_f = selector.fit_transform(train_matrix[:, nonconstant], y_train_f)
X_val_f = selector.transform(val_matrix[:, nonconstant])
assert np.isfinite(X_train_f).all() and np.isfinite(X_val_f).all()
foundation_results = []

# Free the training model and optimizer references before pretrained inference.
if "model_t" in globals():
    model_t.cpu()
for variable in ("optimizer", "scheduler", "loss_fn", "logits", "loss"):
    globals().pop(variable, None)
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

def save_benchmark(result):
    result["package_versions"] = {name: importlib.metadata.version(name) for name in ("tabicl", "tabpfn", "torch", "scikit-learn")}
    result.update(train_rows=len(y_train_f), validation_rows=len(y_val_f),
                  features=X_train_f.shape[1], split="shared_outer_split_small_benchmark", seed=SEED)
    foundation_results[:] = [r for r in foundation_results if r["model"] != result["model"]]
    foundation_results.append(result)
    (ARTIFACT_DIR / "foundation_benchmarks.json").write_text(json.dumps(foundation_results, indent=2))
    print(result)

def benchmark(name, factory):
    candidate = None
    started = time.perf_counter()
    try:
        candidate = factory()
        candidate.fit(X_train_f, y_train_f)
        # Limit query memory; training context remains identical in every batch.
        positive_col = list(candidate.classes_).index(1)
        scores = np.concatenate([candidate.predict_proba(X_val_f[i:i+64])[:, positive_col]
                                 for i in range(0, len(X_val_f), 64)])
        result = {"model": name, "status": "ok",
                  "roc_auc": float(roc_auc_score(y_val_f, scores)),
                  "auprc": float(average_precision_score(y_val_f, scores))}
    except Exception as exc:
        result = {"model": name, "status": "failed", "error": str(exc)[:1500]}
    finally:
        del candidate
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    result["seconds"] = round(time.perf_counter() - started, 2)
    save_benchmark(result)

from lightgbm import LGBMClassifier
benchmark("LightGBM_matched", lambda: LGBMClassifier(n_estimators=300, learning_rate=0.05,
          num_leaves=64, class_weight="balanced", random_state=SEED, verbosity=-1))


In [ ]:
# === Optional benchmark: TabPFN ===
def make_tabpfn():
    # Fail clearly if model access requires login; never wait for a hidden browser.
    os.environ["TABPFN_NO_BROWSER"] = "1"
    from tabpfn import TabPFNClassifier
    return TabPFNClassifier(device=DEVICE, n_estimators=1, random_state=SEED)

benchmark("TabPFN", make_tabpfn)


In [ ]:
# === Optional benchmark: TabICL ===
def make_tabicl():
    from tabicl import TabICLClassifier
    return TabICLClassifier(device=DEVICE, n_estimators=1, batch_size=1, random_state=SEED)

benchmark("TabICL", make_tabicl)


In [ ]:
# === TabFM research status ===
save_benchmark({"model": "TabFM", "status": "not_implemented",
                "reason": "No verified checkpoint integration in this notebook."})


In [ ]:
# === Save foundation benchmark summary ===
for result in foundation_results:
    result["package_versions"] = {name: importlib.metadata.version(name) for name in ("tabicl", "tabpfn", "torch", "scikit-learn")}
with open(ARTIFACT_DIR / "foundation_benchmarks.json", "w") as f:
    json.dump(foundation_results, f, indent=2)
display(pd.DataFrame(foundation_results))


## 6. Download These Outputs

After the notebook finishes, download `/kaggle/working/artifacts`.

Main repo artifacts:

- `model.pt`
- `metrics.json`
- `ft_transformer_encoder.pkl`
- `ft_transformer_scaler.pkl`
- `lightgbm_metrics.json`
- `foundation_benchmarks.json`

Next step in the GitHub repo: update the scorecard and decide whether this checkpoint is good enough for the first deployed API.


## 7. Final Submission

Generate `submission.csv` with the verified Transformer checkpoint, then submit
it as a late entry to IEEE-CIS. The competition evaluates ROC-AUC.

For inference only, attach this notebook's version 1 output and run the core imports,
dataset discovery, GPU check, FTTransformer class definition, and the two cells below.
For a fresh end-to-end experiment, run the whole notebook. Pretrained benchmarks
are separate from the submission model.

The final CSV contains every test TransactionID once, in sample-submission order.
Inference uses batches of 64 and never fits preprocessing on test data.
`results.json` keeps the main validation and small foundation-model comparison separate.
The Kaggle score is recorded only after the site evaluates the submission.


In [ ]:
"""Notebook companion: chunked IEEE-CIS inference and strict CSV validation.

Run the function with the notebook's restored FTTransformer and preprocessing.pkl.
The notebook embeds this file so it does not need a GitHub checkout on Kaggle.
"""
from pathlib import Path
import hashlib
import json

import numpy as np
import pandas as pd


def validate_submission(submission, sample):
    if list(submission.columns) != ["TransactionID", "isFraud"]:
        raise ValueError("Submission must contain TransactionID,isFraud in that order")
    if submission.empty or len(submission) != len(sample):
        raise ValueError("Submission row count differs from sample_submission.csv")
    if not submission["TransactionID"].is_unique:
        raise ValueError("Duplicate transaction IDs")
    if not np.array_equal(submission["TransactionID"].to_numpy(), sample["TransactionID"].to_numpy()):
        raise ValueError("Transaction IDs or their order differ from the sample")
    probabilities = submission["isFraud"].to_numpy(dtype=float)
    if not np.isfinite(probabilities).all() or not ((probabilities >= 0) & (probabilities <= 1)).all():
        raise ValueError("Fraud predictions must be finite probabilities between zero and one")


def prepare_test_chunk(chunk, identity, preprocessing):
    # Test identity columns use id-XX; training used id_XX.
    identity = identity.rename(columns=lambda name: name.replace("id-", "id_"))
    merged = chunk.merge(identity, on="TransactionID", how="left", sort=False, validate="one_to_one")
    if not np.array_equal(merged["TransactionID"], chunk["TransactionID"]):
        raise ValueError("Identity join changed transaction ordering")
    numeric = preprocessing["numeric_features"]
    categorical = preprocessing["categorical_features"]
    missing = set(numeric + categorical) - set(merged.columns)
    if missing:
        raise ValueError(f"Missing model features: {sorted(missing)}")
    xn = preprocessing["scaler"].transform(
        merged[numeric].replace([np.inf, -np.inf], np.nan).fillna(preprocessing["numeric_medians"])
    ).astype("float32")
    xc = preprocessing["encoder"].transform(
        merged[categorical].fillna("__missing__").astype(str)
    ).astype("int64") + 1
    if not np.isfinite(xn).all() or (xc < 0).any():
        raise ValueError("Preprocessing produced invalid numeric values or category indices")
    return xn, xc


def generate_submission(model, preprocessing, data_dir, output_dir, device="cuda", batch_size=64):
    import torch

    data_dir, output_dir = Path(data_dir), Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    sample = pd.read_csv(data_dir / "sample_submission.csv")
    identity = pd.read_csv(data_dir / "test_identity.csv")
    if not identity["TransactionID"].is_unique:
        raise ValueError("Duplicate test identity IDs")
    model = model.to(device).eval()
    all_ids, all_predictions = [], []
    # Fix object dtypes across chunks; do not refit preprocessing on test data.
    category_types = {name: "object" for name in preprocessing["categorical_features"]}
    for chunk in pd.read_csv(data_dir / "test_transaction.csv", chunksize=10000, dtype=category_types):
        xn, xc = prepare_test_chunk(chunk, identity, preprocessing)
        predictions = []
        with torch.inference_mode():
            for offset in range(0, len(chunk), batch_size):
                logits = model(torch.from_numpy(xn[offset:offset+batch_size]).to(device),
                               torch.from_numpy(xc[offset:offset+batch_size]).to(device))
                predictions.append(torch.sigmoid(logits).cpu().numpy())
        all_ids.append(chunk["TransactionID"].to_numpy())
        all_predictions.append(np.concatenate(predictions))
        print(f"Predicted {sum(len(ids) for ids in all_ids):,} / {len(sample):,} rows", flush=True)
    predictions = pd.Series(np.concatenate(all_predictions), index=np.concatenate(all_ids))
    if not predictions.index.is_unique or set(predictions.index) != set(sample["TransactionID"]):
        raise ValueError("Test predictions do not cover exactly the sample transaction IDs")
    submission = sample[["TransactionID"]].copy()
    submission["isFraud"] = submission["TransactionID"].map(predictions)
    validate_submission(submission, sample)
    destination = output_dir / "submission.csv"
    submission.to_csv(destination, index=False)
    validate_submission(pd.read_csv(destination), sample)
    manifest = {
        "model": "ft_transformer_t4_safe", "rows": len(submission),
        "columns": list(submission.columns), "batch_size": batch_size,
        "sha256": hashlib.sha256(destination.read_bytes()).hexdigest(),
        "probability_min": float(submission.isFraud.min()),
        "probability_max": float(submission.isFraud.max()),
        "status": "validated_csv_not_yet_submitted", "training_rows": 40000,
    }
    (output_dir / "submission_manifest.json").write_text(json.dumps(manifest, indent=2))
    print(json.dumps(manifest, indent=2))
    return submission


In [ ]:
# === Restore the verified Transformer and generate the Kaggle submission ===
# Run core imports, dataset discovery, GPU check, and the FTTransformer class cell first.
import shutil
import joblib

# Prefer the current run; otherwise restore this notebook's attached saved output.
if not (ARTIFACT_DIR / "preprocessing.pkl").exists():
    candidates = [path.parent for path in Path("/kaggle/input").rglob("preprocessing.pkl")
                  if "notebook70df114be2" in str(path) and (path.parent / "model.pt").exists()]
    if len(candidates) != 1:
        raise FileNotFoundError("Attach version 1 output of notebook70df114be2, or run the training cells first.")
    for path in candidates[0].iterdir():
        if path.is_file():
            shutil.copy2(path, ARTIFACT_DIR / path.name)
    print("Restored artifacts from:", candidates[0])

checkpoint = torch.load(ARTIFACT_DIR / "model.pt", map_location="cpu", weights_only=True)
preprocessing = joblib.load(ARTIFACT_DIR / "preprocessing.pkl")
submission_model = FTTransformer(
    checkpoint["num_numeric_features"], checkpoint["category_cardinalities"],
    checkpoint["d_model"], checkpoint["nhead"], checkpoint["num_layers"], checkpoint["dropout"],
)
submission_model.load_state_dict(checkpoint["model_state"])
submission = generate_submission(submission_model, preprocessing, DATA_DIR,
                                 "/kaggle/working", device=DEVICE, batch_size=64)

# Bind the CSV to the exact checkpoint used; do not invent a Kaggle score.
manifest_path = Path("/kaggle/working/submission_manifest.json")
manifest = json.loads(manifest_path.read_text())
manifest["checkpoint_sha256"] = hashlib.sha256((ARTIFACT_DIR / "model.pt").read_bytes()).hexdigest()
manifest["validation_metrics"] = json.loads((ARTIFACT_DIR / "metrics.json").read_text())
manifest_path.write_text(json.dumps(manifest, indent=2))

# Collate results into distinct evaluation groups.
main_results = [json.loads((ARTIFACT_DIR / name).read_text())
                for name in ("lightgbm_metrics.json", "metrics.json")]
foundation_path = ARTIFACT_DIR / "foundation_benchmarks.json"
foundation_results = json.loads(foundation_path.read_text()) if foundation_path.exists() else []
results = {"main_validation": main_results, "small_matched_benchmark": foundation_results,
           "kaggle_submission": {"file": "submission.csv", "status": "ready", "score": None}}
Path("/kaggle/working/results.json").write_text(json.dumps(results, indent=2))
print("Main validation: 40,000 training / 10,000 validation rows, 432 features")
display(pd.DataFrame(main_results).drop(columns=["history"], errors="ignore"))
print("Small matched benchmark: 2,000 training / 1,000 validation rows, 100 features")
display(pd.DataFrame(foundation_results))
print("Validated submission.csv is ready. Kaggle scoring is a separate step.")
